# 05 — Entraînement — le pipeline réel, ses artefacts et sa reproductibilité

**Projet** : Routage de tickets de support par un encodeur appris sur le corpus

## Objectifs pédagogiques

1. Exécuter le pipeline d'entraînement dans un bac à sable qui n'écrase rien.
1. Lire les cinq artefacts produits : modèle, prétraitement, métriques, fiche, configuration résolue.
1. Recharger le modèle depuis son artefact et vérifier qu'il prédit **exactement** comme un modèle ajusté sur les mêmes données.
1. Prouver le déterminisme : deux ajustements à graine fixée donnent le même chiffre.

## 1. Mise en place

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (320 tickets) quand `data/raw` est
# vide : l'exécution reste rapide. Si le corpus de référence a été généré par `make data`, il est
# utilisé tel quel — les valeurs absolues d'un notebook ne sont pas celles de la référence.
NB_DOCUMENTS = 320

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks lisent le corpus du projet et écrivent leurs artefacts dans `outputs/notebooks` :
# ils ne doivent jamais écraser ceux produits par `make train` / `make evaluate`.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PATHS.data_dir,
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

MODEL_NODE = dict(node(CONFIG, "model"))
TEXT_COLUMN = str(MODEL_NODE.get("text_column", "text"))
TARGET_COLUMN = str(CONFIG.data.target or "label")
LABELS = [
    "facturation",
    "livraison",
    "produit_defectueux",
    "remboursement",
    "compte_client",
    "autre",
]
STYLES = [
    "canonique",
    "paraphrase",
    "bruite",
]

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (seuil contractuel : 0.7)")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(f"Classes           : {len(LABELS)} — {', '.join(LABELS)}")
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

In [ ]:
from src.data.generators import SyntheticTicketGenerator
from src.data.loaders import TextLabelLoader

LOADER = TextLabelLoader(
    PATHS,
    formats=CONFIG.data.formats,
    validation_enabled=CONFIG.data.validation.raw,
    lazy_validation=CONFIG.data.validation.lazy,
)
if LOADER.documents_path.exists():
    DOCUMENTS = LOADER.load_documents()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis data/raw : {len(DOCUMENTS)} tickets, graine {METADATA.get('seed')}")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté pour que les pipelines (train, evaluate) trouvent les mêmes fichiers que le
    # notebook ; `make data` régénère le corpus de référence.
    bundle = SyntheticTicketGenerator.from_config(CONFIG.data, seed=CONFIG.seed).generate()
    DOCUMENTS, METADATA = bundle.documents, bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_metadata(METADATA)
    print(f"data/raw absent : corpus de {len(DOCUMENTS)} tickets généré et persisté")

TRAIN = LOADER.split("train", frame=DOCUMENTS)
VALIDATION = LOADER.split("val", frame=DOCUMENTS)
CALIBRATION = LOADER.split("calibration", frame=DOCUMENTS)
TEST = LOADER.split("test", frame=DOCUMENTS)
# La calibration sert à **régler** (seuil d'automatisation), la validation à **choisir**
# (algorithme, rééquilibrage), et le test à **juger** — une seule fois, à la fin.
print(
    "splits :",
    {
        "train": len(TRAIN),
        "val": len(VALIDATION),
        "calibration": len(CALIBRATION),
        "test": len(TEST),
    },
)

LABEL_COUNTS = DOCUMENTS[TARGET_COLUMN].value_counts()
MAJORITY_LABEL = str(LABEL_COUNTS.index[0])
MINORITY_LABEL = str(LABEL_COUNTS.index[-1])
print(
    f"classe majoritaire : {MAJORITY_LABEL} ({LABEL_COUNTS.iloc[0] / len(DOCUMENTS):.1%}) "
    f"| classe minoritaire : {MINORITY_LABEL} ({LABEL_COUNTS.iloc[-1] / len(DOCUMENTS):.1%})"
)

## 2. Le pipeline d'entraînement, dans un bac à sable

Le notebook appelle **le pipeline du projet**, pas une réimplémentation : ce qu'on lit ici est exactement ce que fait `python -m src.main mode=train`. Les artefacts partent dans `outputs/notebooks/artifacts`, donc ceux de `make train` ne sont jamais écrasés.

In [ ]:
from src.pipelines import DataGenerationPipeline, TrainPipeline

data_result = DataGenerationPipeline(CONFIG, paths=NB_PATHS).run()
print(
    "generate-data :",
    "succès" if data_result.succeeded else "échec",
    f"en {data_result.duration_seconds:.2f}s",
)
for message in data_result.messages:
    print("  -", message)

train_result = TrainPipeline(CONFIG, paths=NB_PATHS).run()
print()
print(
    "train :",
    "succès" if train_result.succeeded else "échec",
    f"en {train_result.duration_seconds:.2f}s",
)
for message in train_result.messages:
    print("  -", message)
assert train_result.succeeded, train_result.messages
outcome = train_result.payload

In [ ]:
artifacts = {
    "modèle": NB_PATHS.models_dir / "model.pt",
    "prétraitement": NB_PATHS.models_dir / "preprocessing.joblib",
    "métriques": NB_PATHS.metrics_dir / "training_metrics.json",
    "fiche de modèle": NB_PATHS.models_dir / "model_card.json",
    "configuration résolue": NB_PATHS.models_dir / "resolved_config.json",
}
for name, path in artifacts.items():
    size = path.stat().st_size if path.exists() else 0
    print(
        f"{name:24s} {'écrit' if path.exists() else 'ABSENT':6s} "
        f"{path.relative_to(PROJECT_ROOT)} ({size} octets)"
    )
assert all(path.exists() for path in artifacts.values())

**Ce qu'il faut retenir**

- Cinq artefacts, cinq questions : le modèle (comment prédire), le prétraitement (comment préparer le texte), les métriques (ce que ça vaut), la fiche (comment c'est fait) et la configuration résolue (avec quels réglages exactement).
- La configuration résolue est le seul moyen de rejouer un run : la configuration Hydra est faite de couches, et une couche qui change change tout.
- `DataGenerationPipeline` a d'abord persisté le corpus : l'entraînement ne peut pas lire un `data/raw` fantôme.

## 3. Lire les métriques et la fiche du run

Les métriques d'entraînement distinguent ce que le modèle a **vu** (`train_*`) de ce qu'il n'a pas vu (`val_*`). L'écart entre les deux est la première mesure de généralisation disponible.

In [ ]:
import json

training = json.loads((NB_PATHS.metrics_dir / "training_metrics.json").read_text(encoding="utf-8"))
display(pd.Series(training["metrics"]).to_frame("valeur").round(4))

gap = float(training["metrics"]["train_accuracy"] - training["metrics"]["val_accuracy"])
print(
    f"tickets d'entraînement : {training['n_train_documents']} | "
    f"tickets de validation : {training['n_val_documents']}"
)
print(f"écart exactitude train - validation : {gap:+.4f}")
print(
    f"métrique principale du run : val_{CONFIG.metrics.primary}="
    f"{float(training['metrics'][f'val_{CONFIG.metrics.primary}']):.4f} "
    f"(seuil contractuel : {CONFIG.metrics.min_primary})"
)
fit = outcome.fit_result
print(f"ajustement : {fit.n_samples} tickets, {fit.n_features} termes, {fit.duration_seconds:.3f}s")
print(
    "hyper-paramètres effectifs (registre + manifeste) :",
    {"lexical": fit.params.get("lexical"), "estimator": fit.params.get("estimator")},
)

In [ ]:
card = json.loads((NB_PATHS.models_dir / "model_card.json").read_text(encoding="utf-8"))
display(pd.Series({key: str(value) for key, value in card.items()}).to_frame("fiche de modèle"))

**Ce qu'il faut retenir**

- Un écart train / validation nul n'est pas une bonne nouvelle : c'est le signe d'un problème trop simple ou d'une fuite. Ici l'écart est faible parce que la tâche est bien posée, et il est **publié**.
- La fiche de modèle contient les paramètres effectifs, la liste des classes, la taille du vocabulaire et les notes de limitation : c'est elle qui rend le run rejouable par quelqu'un d'autre.
- Les métriques sont écrites en JSON : un portail de CI peut les comparer d'un run à l'autre sans lire un notebook.
- Le pipeline du notebook écrit `data/raw` avec le corpus de la configuration du notebook (__NB_DOCUMENTS__ tickets) : pour relire le corpus de référence, il faut le régénérer par `make data`.

## 4. Recharger le modèle et vérifier l'identité

Un artefact qu'on ne recharge pas est un artefact dont on ne sait rien. On vérifie que le modèle **rechargé** produit les mêmes probabilités qu'un modèle ajusté dans la session — c'est le test qui garantit que l'inférence ne dépend pas du hasard d'un processus.

Comparer l'artefact à un modèle ajusté sur un **autre** corpus ne prouverait rien : l'artefact a été produit par le pipeline sur le corpus de la configuration du notebook, alors que `data/raw` peut contenir le corpus de référence (1 200 tickets, écrit par `make data`). On reconstruit donc le corpus du pipeline — même graine, même configuration, donc mêmes textes — et la comparaison porte sur deux chemins de code, pas sur deux jeux de données.

In [ ]:
from src.data.generators import SyntheticTicketGenerator

PIPELINE_BUNDLE = SyntheticTicketGenerator.from_config(CONFIG.data, seed=CONFIG.seed).generate()
PIPELINE_TRAIN = LOADER.split("train", frame=PIPELINE_BUNDLE.documents)
PIPELINE_VALIDATION = LOADER.split("val", frame=PIPELINE_BUNDLE.documents)
print(
    f"corpus du pipeline : {len(PIPELINE_TRAIN)} tickets d'entraînement, "
    f"{len(PIPELINE_VALIDATION)} en validation"
)
print(
    f"corpus de référence chargé : {len(TRAIN)} / {len(VALIDATION)} "
    "(c'est celui du notebook 01 ; l'artefact, lui, vient du corpus du pipeline)"
)

In [ ]:
from src.models import build_model, load_model

model_path = NB_PATHS.models_dir / "model.pt"
reloaded = load_model(model_path, config=node(CONFIG, "model"))
print("modèle rechargé :", reloaded.summary())

memory_model = build_model(CONFIG)
memory_model.fit(PIPELINE_TRAIN, target=TARGET_COLUMN)

texts = TEST[TEXT_COLUMN].astype(str).tolist()
same = np.allclose(memory_model.predict_proba(texts), reloaded.predict_proba(texts), atol=1e-9)
print("probabilités identiques entre modèle ajusté et artefact rechargé :", same)
assert same, "l'artefact rechargé ne reproduit pas le modèle ajusté"

display(
    pd.DataFrame(
        {
            "texte": [text[:64] + "…" for text in texts[:3]],
            "prédit (mémoire)": memory_model.predict(texts[:3]),
            "prédit (artefact)": reloaded.predict(texts[:3]),
        }
    )
)

## 5. Déterminisme : deux runs, le même chiffre

Le dépôt impose des graines partout. La preuve la plus simple : refaire le run et comparer la métrique principale **exactement** — puis la comparer à celle qu'a enregistrée le pipeline.

In [ ]:
from src.training.metrics import classification_metrics

validation_truth = PIPELINE_VALIDATION[TARGET_COLUMN].astype(str)
second_model = build_model(CONFIG)
second_model.fit(PIPELINE_TRAIN, target=TARGET_COLUMN)

first_scores = memory_model.predict_proba(PIPELINE_VALIDATION[TEXT_COLUMN].astype(str).tolist())
second_scores = second_model.predict_proba(PIPELINE_VALIDATION[TEXT_COLUMN].astype(str).tolist())
first_metrics = classification_metrics(
    validation_truth,
    np.asarray(memory_model.labels)[first_scores.argmax(axis=1)],
    labels=list(memory_model.labels),
)
second_metrics = classification_metrics(
    validation_truth,
    np.asarray(second_model.labels)[second_scores.argmax(axis=1)],
    labels=list(second_model.labels),
)
pipeline_metric = float(training["metrics"][f"val_{CONFIG.metrics.primary}"])
print(
    f"premier ajustement  : {CONFIG.metrics.primary}={first_metrics[CONFIG.metrics.primary]:.10f}"
)
print(
    f"second ajustement   : {CONFIG.metrics.primary}={second_metrics[CONFIG.metrics.primary]:.10f}"
)
print(f"pipeline enregistré : {CONFIG.metrics.primary}={pipeline_metric:.10f}")
assert first_metrics[CONFIG.metrics.primary] == second_metrics[CONFIG.metrics.primary]
assert abs(first_metrics[CONFIG.metrics.primary] - pipeline_metric) < 1e-12
print("déterminisme vérifié : même graine, même corpus, même chiffre")

**Ce qu'il faut retenir**

- Le déterminisme n'est pas un luxe pédagogique : c'est ce qui permet de comparer deux variantes de stack sur un écart de F1 et non sur du bruit d'exécution.
- Le modèle rechargé sert dans le pipeline d'évaluation et dans l'inférence : les trois chemins partagent le même objet, et donc les mêmes probabilités.
- Aucun accès réseau n'est nécessaire à aucune de ces étapes : tout est appris sur le corpus synthétique, et le vocabulaire est appris sur le train.